# Step 7 on the test: robustness checks

**Question.** Are the differences of [Step 6 on the test](06_test_backtest.ipynb) between B, C and A real or luck? The same checks as the train ([07_robustness.ipynb](07_robustness.ipynb)), decided in [docs/Test.md](../docs/Test.md):

1. final table: metrics, difference from A with a confidence interval (bootstrap on the days), beta and alpha against SPY;
2. costs 0 and 5;
3. dependence on a pair or a sector, and result without PYPL-XYZ.

Each number is next to its train value. The comparison between the two halves of the period and the check without 13 August, which is a train day, **are not done**.

The notebook reads only the files saved in `Results/test/` by [test_final.py](test_final.py) and the SPY prices of the test (`python -m src.test_data`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # project folder: makes src importable

import pandas as pd

from src.config import RESULTS, RESULTS_TEST, SPY_TEST, TEST_START, TEST_END
from src.backtest import load_trades, add_costs, daily_pnl, summary, save_results
from src.signals import list_pairs, load_pair
from src.agent_variants.comparison import Variant, daily_delta, bootstrap, beta_alpha, sectors

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
A = load_trades(RESULTS_TEST / "trades_A.csv", allow_test=True)
B = load_trades(RESULTS_TEST / "trades_B.csv", allow_test=True)
C = load_trades(RESULTS_TEST / "trades_C.csv", allow_test=True)
d0 = load_pair(list_pairs().pair.iloc[0])
days = pd.DatetimeIndex(d0.ts.normalize().unique())
days = days[(days >= TEST_START) & (days <= TEST_END)]
ref = {"A": A}
variants = [Variant("finbert_episode", v, "A", True, "FinBERT, episode window", t) for v, t in (("B", B), ("C", C))]
spy = pd.read_parquet(SPY_TEST).set_index("date").close
mkt = spy.pct_change().reindex(days)                 # the return of 17 September uses the close of the 16th
assert mkt.notna().all(), "SPY is missing on some test day"
key = lambda v: (v.name, v.version)                   # noqa: E731
print(f"{len(days)} test days; SPY: return over the period {(spy.iloc[-1] / spy.iloc[0] - 1):.1%}, annual volatility {mkt.std() * 252 ** 0.5:.1%}")

## 1. Final table (cost 0)

- metrics of A, B and C;
- `delta_usd`: dollar difference from A, with the 95% confidence interval obtained by resampling the test days with replacement (10,000 times);
- `share_below_zero`: the share of resamples in which the version earns less than A;
- beta and annualised alpha against SPY, with the t statistics.

The bootstrap treats the days as independent. With 32 days the intervals are wide.

In [ ]:
def metrics(t):
    s = summary(add_costs(t, 0), days)
    return {k: s[k] for k in ("trade", "net_bps", "win_rate", "profit_factor", "avg_minutes", "pnl_usd", "sharpe")} | dict(max_drawdown_pct=100 * s["max_drawdown"]) | \
           beta_alpha(daily_pnl(add_costs(t, 0), days).ret, mkt)

rows = [dict(variant="A", version="") | metrics(A)]
rows += [dict(variant=v.name, version=v.version) | metrics(v.trades) | bootstrap(daily_delta(v, ref, days)) for v in variants]
final = pd.DataFrame(rows).set_index(["variant", "version"])
train = pd.read_csv(RESULTS / "robustness/final_table.csv", index_col=[0, 1]).rename(index=lambda x: "" if pd.isna(x) else x, level=1)
train = train.loc[[("A", ""), ("finbert_episode", "B"), ("finbert_episode", "C")]]
cols = ["trade", "net_bps", "win_rate", "profit_factor", "sharpe", "max_drawdown_pct", "pnl_usd", "delta_usd", "ci95_low", "ci95_high", "share_below_zero",
        "beta", "t_beta", "alpha_annual", "t_alpha"]
pd.concat({"test": final[cols], "train": train[cols]}, names=["period"])

## 2. Costs

Dollar difference from A with 0 and 5 basis points per operation. Every blocked trade saves its costs: with high costs even a random block improves the result, so an advantage that shows up only at cost 5 does not mean that the agent chooses well.

In [ ]:
COSTS = (0, 5)
costs = pd.DataFrame({key(v): {c: summary(add_costs(v.trades, c), days)["pnl_usd"] - summary(add_costs(A, c), days)["pnl_usd"] for c in COSTS}
                      for v in variants}).T.rename_axis(["variant", "version"])
costs.columns = [f"delta_usd_cost_{c}" for c in COSTS]
costs_train = pd.read_csv(RESULTS / "robustness/costs.csv", index_col=[0, 1]).loc["finbert_episode"][costs.columns]
pd.concat({"test": costs, "train": pd.concat({"finbert_episode": costs_train})}, names=["period"])

## 3. Does it depend on a pair or a sector?

Dollar difference from A (cost 0) per pair and per sector, next to the train. Then the result without PYPL-XYZ, the pair from which almost all the difference came on the train.

In [ ]:
sector = sectors()

def delta_by_pair(v):
    p = lambda t: add_costs(t, 0).groupby("pair").pnl_usd.sum()   # noqa: E731
    return p(v.trades).sub(p(A), fill_value=0).reindex(sector.index, fill_value=0)

by_pair = pd.DataFrame({key(v): delta_by_pair(v) for v in variants}).T.rename_axis(["variant", "version"])
pairs_train = pd.read_csv(RESULTS / "robustness/pairs.csv", index_col=[0, 1]).loc[["finbert_episode"]]
pd.concat({"test": by_pair, "train": pairs_train[by_pair.columns]}, names=["period"]).T

In [ ]:
by_sector = by_pair.T.groupby(sector).sum().T
sectors_train = pd.read_csv(RESULTS / "robustness/sectors.csv", index_col=[0, 1]).loc[["finbert_episode"]]
pd.concat({"test": by_sector, "train": sectors_train.reindex(columns=by_sector.columns)}, names=["period"]).T

In [ ]:
without = pd.DataFrame({"delta_usd": by_pair.sum(axis=1), "delta_PYPL_XYZ": by_pair["PYPL-XYZ"], "delta_without_PYPL_XYZ": by_pair.drop(columns="PYPL-XYZ").sum(axis=1)})
without_train = pd.DataFrame({"delta_usd": pairs_train.sum(axis=1), "delta_PYPL_XYZ": pairs_train["PYPL-XYZ"],
                              "delta_without_PYPL_XYZ": pairs_train.drop(columns="PYPL-XYZ").sum(axis=1)})
pd.concat({"test": without, "train": without_train}, names=["period"])

## 4. Saving

In [ ]:
for p in save_results({"test/robustness/final_table": final, "test/robustness/costs": costs, "test/robustness/pairs": by_pair,
                       "test/robustness/sectors": by_sector, "test/robustness/without_PYPL_XYZ": without}):
    print(p.relative_to(RESULTS.parent))

## 5. What the checks tell us

**B: indistinguishable from A.** B loses 189 dollars, with a 95% interval **from -527 to +135**: it contains zero, so the difference can be luck. It is below A in 87% of the resamples. On the train the interval was from -1,598 to +834.

**C: worse than A.** C loses 1,242 dollars, with an interval **from -2,183 to -374**: it is entirely below zero and C is below A in all the resamples. On the test following the news worsens the strategy with some confidence. On the train the interval still contained zero (from -2,202 to +847).

**Market.** The beta of A, B and C against SPY is -0.04, with a t statistic around -2.2. In a period in which SPY rose by 0.5%, it means a very small link with the stock market. It stays an almost market-neutral strategy, as on the train (beta 0.01). The annualised alpha of A has t = 1.9 (train 4.1): over 32 days scaled to a year it must be read with caution.

**Costs.** At 5 basis points B loses only 49 dollars relative to A: by blocking 7 trades it saves their costs. It is a cost saving, not a better choice. C stays 1,142 dollars below.

**Pairs and sectors.**
- **B.** The 7 blocks are in three pairs: ISRG-SYK (-102), BAC-C (-63) and DHR-TMO (-25), plus +2 on CRM-WDAY. On the test the agent blocks nothing on PYPL-XYZ. Removing that pair does not change B (-189), while on the train it made it positive (+279). The explanation of the train ("the difference comes from a single trade on PYPL-XYZ") does not hold on the test: B loses on several pairs.
- **C.** It loses in 9 pairs out of 15 and earns in 4. Without PYPL-XYZ it still loses 852 dollars (train +832). The loss is spread out, not tied to one pair.

**In short.** The checks confirm Step 6 on the test. On the test B cannot be told apart from A and C is worse than A. Neither blocking nor following the news with `finbert_episode` improves the pairs-trading strategy.

**Limits.**
- The bootstrap treats the days as independent, and there are only 32 days.
- The intervals are narrower than on the train because the period is shorter and A earns less, not because the agent is more predictable.